### Импорты библиотек и загрузка данных

In [3]:
import os, gc, re, warnings
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

C:\Users\fok55\PycharmProjects\for_all_things\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
train = pd.read_parquet('train.parquet')
bench_queries = pd.read_parquet('benchmark_queries.parquet')
bench_items = pd.read_parquet('benchmark_items.parquet')

for df in [train, bench_items]:
    for col in ['item_price', 'item_longitude', 'item_latitude']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

train.shape, bench_queries.shape, bench_items.shape

((497673, 19), (2452, 6), (189212, 14))

### Функции для подготовки текстовых полей для поиска 

In [5]:
WORD_RE = re.compile(r'[а-яёa-z0-9]{2,}')
def norm(s):     return str(s).lower() if s is not None else ''
def token_set(s):return set(WORD_RE.findall(str(s).lower()))

def build_query_text(df):
    return (df['search_query'].fillna('').apply(norm) + ' ' +
            df['search_infm_params_text'].fillna('').apply(norm)).str.strip()

def build_item_text(df):
    return (df['item_title_raw'].fillna('').apply(norm) + ' ' +
            df['item_description_raw'].fillna('').apply(norm) + ' ' +
            df['item_infm_params_text'].fillna('').apply(norm)).str.strip()

train['_q_text'] = build_query_text(train)
train['_q_norm'] = train['search_query'].str.lower().str.strip()
train['_i_text'] = build_item_text(train)
train['_i_title'] = train['item_title_raw'].fillna('').apply(norm)

bench_queries['_q_text'] = build_query_text(bench_queries)
bench_queries['_q_norm'] = bench_queries['search_query'].str.lower().str.strip()
bench_items['_i_text']   = build_item_text(bench_items)
bench_items['_i_title']  = bench_items['item_title_raw'].fillna('').apply(norm)


bench_items_set = set(bench_items['item_id'])
bench_q_norms   = set(bench_queries['_q_norm'].dropna().unique())
mem_pairs = train[train['_q_norm'].isin(bench_q_norms) & train['item_id'].isin(bench_items_set)][['_q_norm','item_id']].drop_duplicates()
memory_map = defaultdict(list)
for qn, iid in mem_pairs.itertuples(index=False):
    memory_map[qn].append(iid)
bench_queries['_memory_items'] = bench_queries['_q_norm'].map(lambda t: memory_map.get(t, []))

'память:', (bench_queries['_memory_items'].apply(len) > 0).sum(), 'запросов,', bench_queries['_memory_items'].apply(len).sum(), 'items'

('память:', np.int64(360), 'запросов,', np.int64(940), 'items')

### Tf-Idf с полным текстом и заголовками

In [6]:
vec_full = TfidfVectorizer(
    max_features=80_000,      
    ngram_range=(1,2), min_df=3, max_df=0.9,
    sublinear_tf=True, dtype=np.float32,
)
vec_full.fit(bench_items['_i_text'])

item_tfidf_bench = vec_full.transform(bench_items['_i_text'])
query_tfidf_bench = vec_full.transform(bench_queries['_q_text'])
train_i_tfidf = vec_full.transform(train['_i_text'])
train_q_tfidf = vec_full.transform(train['_q_text'])

vec_title = TfidfVectorizer(
    max_features=30_000, ngram_range=(1,2),
    min_df=3, max_df=0.9, sublinear_tf=True, dtype=np.float32,
)
vec_title.fit(bench_items['_i_title'])
item_tfidf_title_bench  = vec_title.transform(bench_items['_i_title'])
query_tfidf_title_bench = vec_title.transform(bench_queries['_q_norm'])

print('full: ', item_tfidf_bench.shape)
print('title:', item_tfidf_title_bench.shape)

full:  (189212, 80000)
title: (189212, 30000)


### Генерация топ-К кандидатов с помощью tf-idf

In [7]:
def topk_candidates(query_tfidf, item_tfidf, K, batch_size=64):
    n = query_tfidf.shape[0]
    all_idx = np.zeros((n, K), dtype=np.int32)
    all_scores = np.zeros((n, K), dtype=np.float32)
    item_T = item_tfidf.T.tocsr()
    for s in tqdm(range(0, n, batch_size), desc='topK'):
        e = min(s + batch_size, n)
        sim = (query_tfidf[s:e] @ item_T).toarray()
        idx_p = np.argpartition(-sim, K - 1, axis=1)[:, :K]
        rows = np.arange(e - s)[:, None]
        sc_p = sim[rows, idx_p]
        order = np.argsort(-sc_p, axis=1)
        all_idx[s:e] = idx_p[rows, order]
        all_scores[s:e] = sc_p[rows, order]
    return all_idx, all_scores

K_FULL = 8000
K_TITLE = 3000

bench_idx_full,  bench_scores_full  = topk_candidates(query_tfidf_bench, item_tfidf_bench, K_FULL)
bench_idx_title, bench_scores_title = topk_candidates(query_tfidf_title_bench, item_tfidf_title_bench, K_TITLE)

bench_items_arr = bench_items['item_id'].values
hit_full = hit_title = hit_union = total_mem = 0
for i, row in bench_queries.iterrows():
    mem = set(row['_memory_items'])
    if not mem: continue
    full_set = set(bench_items_arr[bench_idx_full[i]])
    title_set = set(bench_items_arr[bench_idx_title[i]])
    total_mem += len(mem)
    hit_full += len(mem & full_set)
    hit_title += len(mem & title_set)
    hit_union += len(mem & (full_set | title_set))
print(f'хиты в память: по тоталам = {total_mem} | по фуллам = {hit_full} ({100*hit_full/total_mem:.1f}%) | '
      f'по объединениям = {hit_union} ({100*hit_union/total_mem:.1f}%)')

topK: 100%|██████████| 39/39 [00:01<00:00, 22.64it/s]


хиты в память: по тоталам = 940 | по фуллам = 892 (94.9%) | по объединениям = 894 (95.1%)


### Строим обучающий датасет 

In [8]:
np.random.seed(42)

pos = train.copy().reset_index(drop=True)
pos['label'] = 1

rand_ids = np.random.choice(train['item_id'].unique(), size=len(pos), replace=True)
item_meta = (train.drop_duplicates('item_id').set_index('item_id')
             [['_i_text','item_category_id','item_location_id','item_price',
               'item_rating','item_rating_reviews_count',
               'item_infm_params_text','item_title_raw','item_description_raw']])
neg_item_meta = item_meta.loc[rand_ids].reset_index(drop=True)

neg = pos.copy()
neg['item_id'] = rand_ids
for col in neg_item_meta.columns:
    neg[col] = neg_item_meta[col].values
neg['label'] = 0

mask_same = (neg['item_id'].values == pos['item_id'].values)
pos_c = pos[~mask_same].copy()
neg_c = neg[~mask_same].copy()

df_lgb = pd.concat([pos_c, neg_c], ignore_index=True)\
           .sample(frac=1.0, random_state=42).reset_index(drop=True)

print('df_lgb:', df_lgb.shape)
print('pos:', (df_lgb['label']==1).sum(), '| neg:', (df_lgb['label']==0).sum())
del pos, neg, pos_c, neg_c, neg_item_meta
gc.collect()

df_lgb: (995344, 24)
pos: 497672 | neg: 497672


59

### Подбираем фичи для обучения реранкера

In [9]:
q_tf = vec_full.transform(df_lgb['_q_text'].fillna(''))
i_tf = vec_full.transform(df_lgb['_i_text'].fillna(''))
df_lgb['tfidf_cos_full'] = np.asarray(q_tf.multiply(i_tf).sum(axis=1)).ravel()
del q_tf, i_tf; gc.collect()

title_map_bench = bench_items.drop_duplicates('item_id').set_index('item_id')['_i_title'].to_dict()
title_map_train = train.drop_duplicates('item_id').set_index('item_id')['_i_title'].to_dict()
df_lgb['_i_title'] = df_lgb['item_id'].map(title_map_bench)\
                                       .fillna(df_lgb['item_id'].map(title_map_train))\
                                       .fillna('')
q_t = vec_title.transform(df_lgb['_q_norm'].fillna(''))
i_t = vec_title.transform(df_lgb['_i_title'].fillna(''))
df_lgb['tfidf_cos_title'] = np.asarray(q_t.multiply(i_t).sum(axis=1)).ravel()
del q_t, i_t; gc.collect()

df_lgb['loc_match'] = (df_lgb['search_location_id'] == df_lgb['item_location_id']).astype(np.int8)
df_lgb['cat_match'] = (df_lgb['search_category']    == df_lgb['item_category_id']).astype(np.int8)
df_lgb['price_log'] = np.log1p(df_lgb['item_price'].clip(lower=0).fillna(0))
df_lgb['rating'] = df_lgb['item_rating'].fillna(-1)
df_lgb['reviews_log'] = np.log1p(df_lgb['item_rating_reviews_count'].fillna(0))
df_lgb['phone_hidden'] = df_lgb['item_is_phone_hidden'].astype(np.int8)
df_lgb['msg_forbidden'] = df_lgb['item_is_message_forbidden'].astype(np.int8)
df_lgb['title_len'] = df_lgb['item_title_raw'].fillna('').str.len().clip(0, 500).astype(np.int16)
df_lgb['desc_len'] = df_lgb['item_description_raw'].fillna('').str.len().clip(0, 3000).astype(np.int16)
df_lgb['has_rating'] = df_lgb['item_rating'].notna().astype(np.int8)

filt = df_lgb['search_infm_params_text'].fillna('').astype(str).str.strip().str.lower().values
iprm = df_lgb['item_infm_params_text'].fillna('').astype(str).str.lower().values
fm = np.zeros(len(df_lgb), dtype=np.int8)
for k in range(len(df_lgb)):
    if filt[k] and filt[k] in iprm[k]:
        fm[k] = 1
df_lgb['filter_match'] = fm

q_tok = [token_set(t) for t in df_lgb['_q_text'].tolist()]
i_tok = [token_set(t) for t in df_lgb['_i_text'].tolist()]
jacc  = np.zeros(len(df_lgb), dtype=np.float32)
inter = np.zeros(len(df_lgb), dtype=np.int16)
for k in range(len(df_lgb)):
    a, b = q_tok[k], i_tok[k]
    if a and b:
        ii = len(a & b)
        inter[k] = ii
        jacc[k]  = ii / len(a | b)
df_lgb['jaccard'] = jacc
df_lgb['inter_cnt'] = inter
del q_tok, i_tok; gc.collect()

features = ['tfidf_cos_full','tfidf_cos_title','jaccard','inter_cnt',
            'loc_match','cat_match','price_log','rating','reviews_log',
            'phone_hidden','msg_forbidden','filter_match',
            'title_len','desc_len','has_rating']

### Обучаем Lightgmb

In [10]:
df_lgb['_group'] = (df_lgb['search_query'].astype(str) + '|' +
                    df_lgb['search_location_id'].astype(str) + '|' +
                    df_lgb['search_category'].astype(str))
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
tr_idx, va_idx = next(gss.split(df_lgb, groups=df_lgb['_group']))

X_tr = df_lgb.iloc[tr_idx][features]; y_tr = df_lgb.iloc[tr_idx]['label']
X_va = df_lgb.iloc[va_idx][features]; y_va = df_lgb.iloc[va_idx]['label']

params = {'objective':'binary','metric':'auc','learning_rate':0.05,'num_leaves':63,
          'min_data_in_leaf':200,'feature_fraction':0.8,'bagging_fraction':0.8,
          'bagging_freq':5,'verbose':-1,'seed':42}
model = lgb.train(params, lgb.Dataset(X_tr, y_tr), num_boost_round=500,
                  valid_sets=[lgb.Dataset(X_va, y_va)],
                  callbacks=[lgb.early_stopping(30), lgb.log_evaluation(100)])
print('лучшая итерация:', model.best_iteration)
print(f'auc: {model.best_score["valid_0"]["auc"]:.6f}')

imp = pd.Series(model.feature_importance('gain'), index=features).sort_values(ascending=False)
'важные фичи', imp

Training until validation scores don't improve for 30 rounds
[100]	valid_0's auc: 0.996745
[200]	valid_0's auc: 0.996883
Early stopping, best iteration is:
[228]	valid_0's auc: 0.99689
лучшая итерация: 228
auc: 0.996890


('важные фичи',
 tfidf_cos_full     5.773471e+06
 loc_match          1.969084e+06
 tfidf_cos_title    5.407868e+05
 inter_cnt          2.268670e+05
 filter_match       9.704475e+04
 jaccard            9.496322e+04
 desc_len           3.748667e+04
 price_log          1.609804e+04
 reviews_log        1.450704e+04
 title_len          1.046177e+04
 rating             5.890954e+03
 phone_hidden       3.665558e+03
 msg_forbidden      6.707848e+02
 has_rating         7.591963e+01
 cat_match          0.000000e+00
 dtype: float64)

### Результат модели и создание файла с ответами

In [11]:
n_queries = len(bench_queries)

# --- Union кандидатов full + title ---
cand_q, cand_i = [], []
cand_cos_full, cand_cos_title = [], []
for qi in range(n_queries):
    d = {}
    for idx, sc in zip(bench_idx_full[qi], bench_scores_full[qi]):
        d[int(idx)] = [float(sc), 0.0]
    for idx, sc in zip(bench_idx_title[qi], bench_scores_title[qi]):
        idx = int(idx); sc = float(sc)
        if idx in d: d[idx][1] = sc
        else:        d[idx] = [0.0, sc]
    items = np.fromiter(d.keys(),  dtype=np.int32)
    cf = np.fromiter((d[i][0] for i in d), dtype=np.float32)
    ct = np.fromiter((d[i][1] for i in d), dtype=np.float32)
    cand_q.append(np.full(len(items), qi, dtype=np.int32))
    cand_i.append(items); cand_cos_full.append(cf); cand_cos_title.append(ct)

cand_q = np.concatenate(cand_q)
cand_i = np.concatenate(cand_i)
cand_cos_full = np.concatenate(cand_cos_full)
cand_cos_title = np.concatenate(cand_cos_title)
N = len(cand_q)
print('всего пар-кандидатов:', N)

title_len_all = bench_items['item_title_raw'].fillna('').str.len()\
                    .clip(0, 500).astype(np.int16).values
desc_len_all  = bench_items['item_description_raw'].fillna('').str.len()\
                    .clip(0, 3000).astype(np.int16).values
has_rating_all = bench_items['item_rating'].notna().astype(np.int8).values

q_loc  = bench_queries['search_location_id'].values[cand_q]
q_cat  = bench_queries['search_category'].values[cand_q]
q_filt = bench_queries['search_infm_params_text'].fillna('').astype(str).str.strip().str.lower().values[cand_q]

i_loc   = bench_items['item_location_id'].values[cand_i]
i_cat   = bench_items['item_category_id'].values[cand_i]
i_price = bench_items['item_price'].astype(float).fillna(0).values[cand_i]
i_rate  = bench_items['item_rating'].fillna(-1).values[cand_i]
i_rev   = bench_items['item_rating_reviews_count'].fillna(0).values[cand_i]
i_phone = bench_items['item_is_phone_hidden'].astype(np.int8).values[cand_i]
i_msg   = bench_items['item_is_message_forbidden'].astype(np.int8).values[cand_i]
i_prm   = bench_items['item_infm_params_text'].fillna('').astype(str).str.lower().values[cand_i]

loc_m   = (q_loc == i_loc).astype(np.int8)
cat_m   = (q_cat == i_cat).astype(np.int8)
price_l = np.log1p(np.clip(i_price, 0, None))
rev_l   = np.log1p(i_rev)

t_len = title_len_all[cand_i]
d_len = desc_len_all[cand_i]
has_r = has_rating_all[cand_i]

fm = np.zeros(N, dtype=np.int8)
ne_idx = np.where(q_filt != '')[0]
for k in ne_idx:
    if q_filt[k] in i_prm[k]: fm[k] = 1

item_tok  = [token_set(t) for t in bench_items['_i_text'].tolist()]
query_tok = [token_set(t) for t in bench_queries['_q_text'].tolist()]
jacc  = np.zeros(N, dtype=np.float32)
inter = np.zeros(N, dtype=np.int16)
for k in range(N):
    a = query_tok[cand_q[k]]; b = item_tok[cand_i[k]]
    if a and b:
        ii = len(a & b); inter[k] = ii; jacc[k] = ii / len(a | b)
del item_tok, query_tok; gc.collect()

X = np.column_stack([cand_cos_full, cand_cos_title, jacc, inter, loc_m, cat_m,
                     price_l, i_rate, rev_l, i_phone, i_msg, fm,
                     t_len, d_len, has_r]).astype(np.float32)
scores = model.predict(X, num_iteration=model.best_iteration)
del X; gc.collect()

order = np.lexsort((-scores, cand_q))
sorted_q = cand_q[order]
sorted_i = cand_i[order]

mem_sets     = [set(m) for m in bench_queries['_memory_items'].values]
bench_qid    = bench_queries['query_id'].values
item_ids_arr = bench_items['item_id'].values

bounds = np.searchsorted(sorted_q, np.arange(n_queries + 1))

final = []
for qi in range(n_queries):
    top_idx = sorted_i[bounds[qi]:bounds[qi+1]][:100]
    result  = []; seen = set()
    for iid in mem_sets[qi]:
        if iid not in seen:
            result.append(iid); seen.add(iid)
            if len(result) >= 50: break
    if len(result) < 50:
        for ii in top_idx:
            iid = item_ids_arr[ii]
            if iid not in seen:
                result.append(iid); seen.add(iid)
                if len(result) >= 50: break
    final.append((bench_qid[qi], result))


pd.DataFrame({
    'query_id': [q for q, _ in final],
    'answer':   [' '.join(r) for _, r in final],
}).to_csv('answer.csv', index=False)
print('saved answer.csv')

всего пар-кандидатов: 23620981
saved answer.csv
